# E2 · Localización YOLO antes de CLIP

**Objetivo.** Medir si localizar la mascota antes de extraer su representación mejora la recuperación individual.

**Pregunta.** ¿El recorte automático del animal reduce la influencia del fondo frente a E1, manteniendo sin cambios el encoder y el protocolo?

**Hipótesis descriptiva.** YOLO beneficiará consultas donde el animal ocupa una fracción pequeña de la imagen, aunque errores de detección o recortes agresivos pueden perjudicar casos individuales.

**Intervención y control.** E2 agrega exclusivamente un detector YOLOv8n congelado antes del mismo CLIP ViT-B/32 congelado de E1. El runner recalcula E1 sobre exactamente las mismas consultas y galerías. No hay entrenamiento ni selección de checkpoint.

La tarea es recuperación *closed-set*, de la misma especie y *leave-one-photo-out*. Rank-k agrega por identidad mediante el máximo score entre sus fotos; mAP ordena fotografías. Los dos dominios se evalúan y publican por separado.

### Entorno

Ejecuta el cuaderno en una sesión nueva de Colab con Python 3.11–3.13. La release exacta se instala desde PyPI; no se clona el repositorio, no se copian requirements y no se monta Drive.

In [ ]:
import sys

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("E2 requires Python 3.11–3.13")
%pip install -q --no-cache-dir "pettrace-ml[notebooks,tracking]==0.2.34"

import mlflow
print("MLflow ready:", mlflow.__version__)

In [ ]:
import os
import pettrace
from google.colab import userdata

EXPECTED_PACKAGE = "0.2.34"
LOG_TO_MLFLOW = True
if pettrace.__version__ != EXPECTED_PACKAGE:
    raise RuntimeError(f"Expected PetTrace {EXPECTED_PACKAGE}, found {pettrace.__version__}")

os.environ["MLFLOW_TRACKING_URI"] = "https://ml.pettrace.app"
for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    value = userdata.get(name)
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Missing Colab Secret: {name}")
    os.environ[name] = value
print(f"PetTrace {pettrace.__version__} · MLflow publication required")

## Fase 1. Preparación y preprocesamiento de datos

E2 consume el mismo test público refinado v2 y PetTrace v5 que E1. El runner fija commits y fingerprints, verifica los archivos y rechaza imágenes idénticas compartidas entre dominios. La limpieza, deduplicación y división ocurrieron antes de este experimento.

El único preprocesamiento nuevo es el detector YOLO: conserva la caja de gato o perro de mayor confianza, añade 10 % de margen y usa la imagen completa cuando no existe una caja válida. El reporte guarda la decisión por foto y ejemplos del tensor real entregado a CLIP.

In [ ]:
from IPython.display import Markdown, display
from pettrace.data.refined_sources import RELEASES
from pettrace.experiments.e1 import PRIVATE_COMMIT, PRIVATE_REPO, PRIVATE_VERSION
from pettrace.ui import markdown_table

DATASET_COMMIT = "1777734dbffda0df0bf43305233809ee73157d94"
assert DATASET_COMMIT == PRIVATE_COMMIT
display(Markdown(markdown_table(
    ["Dominio", "Repositorio", "Versión", "Commit", "Uso"],
    [
        ["Público · gatos", RELEASES["lcw"]["repo_id"], "v2", RELEASES["lcw"]["v2_commit"][:12], "test inmutable"],
        ["Público · perros", RELEASES["dogs-world"]["repo_id"], "v2", RELEASES["dogs-world"]["v2_commit"][:12], "test inmutable"],
        ["PetTrace", PRIVATE_REPO, PRIVATE_VERSION, PRIVATE_COMMIT[:12], "evaluación externa"],
    ],
)))

## Fase 2. Representación y entrenamiento

YOLOv8n (Ultralytics) se utiliza como localizador congelado y CLIP ViT-B/32 conforme a [Radford et al. (2021)](https://proceedings.mlr.press/v139/radford21a.html) como encoder congelado. E2 no ajusta pesos: no hay optimizador, épocas, semillas ni selección mediante test.

El runner exige los hashes SHA-256 registrados, reproduce el *golden set* y comprueba que las operaciones CLIP de E1 y E2 sean idénticas. Así, el contraste pareado atribuye la diferencia observada únicamente a la localización.

In [ ]:
from pettrace.experiments.e2 import (
    BASELINE_MODEL, DETECTOR_SETTINGS, DETECTOR_SHA256, ENCODER_SHA256, MODEL,
)

display(Markdown(markdown_table(
    ["Componente", "Condición E2"],
    [
        ["Modelo E2", MODEL],
        ["Control E1", BASELINE_MODEL],
        ["Encoder SHA-256", ENCODER_SHA256],
        ["Detector SHA-256", DETECTOR_SHA256],
        ["Confianza / margen", f"{DETECTOR_SETTINGS['conf']} / {DETECTOR_SETTINGS['margin']:.0%}"],
        ["Fallback", DETECTOR_SETTINGS["fallback"]],
        ["Parámetros entrenados", "0"],
    ],
)))

## Fase 3. Evaluación y análisis de resultados

La siguiente celda ejecuta el flujo oficial: verifica ambos snapshots y modelos, audita el detector, calcula E2 y su control E1 pareado, genera Rank-1/Rank-2/Rank-5/mAP y figuras, y publica **obligatoriamente** dos runs E2 separados en MLflow.

La evidencia local se guarda antes de finalizar la publicación. Ante un fallo de MLflow, reintenta sin inferencia con `pettrace run e2 --resume <reporte>`. La corrida solo está completa cuando `publication.json` contiene `published: true`.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from pettrace.experiments.e2 import run

if not LOG_TO_MLFLOW:
    raise RuntimeError("MLflow publication is mandatory for E2")
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
REPORT = Path("reports") / f"e2-{run_id}"
publication = run(REPORT)
if not publication.get("published"):
    raise RuntimeError("E2 is incomplete: MLflow publication did not finish")

In [ ]:
import json
from IPython.display import Image, Markdown, display

display(Markdown((REPORT / "benchmark-comparison.md").read_text()))
display(Image(filename=str(REPORT / "benchmark-comparison.png")))

for label, folder in (("Test público refinado v2", REPORT / "public_test"), ("PetTrace v5", REPORT)):
    display(Markdown(f"### {label}: contraste pareado E1 → E2"))
    display(Markdown((folder / "e1-e2-comparison.md").read_text()))
    display(Image(filename=str(folder / "e1-e2-comparison.png")))
    summary = json.loads((folder / "summary.json").read_text())
    metrics = summary["global"]
    display(Markdown(
        f"**E2 Rank-1:** {metrics['rank1']:.2%} · **Rank-2:** {metrics['rank2']:.2%} · "
        f"**Rank-5:** {metrics['rank5']:.2%} · **mAP:** {metrics['mAP']:.4f}"
    ))
    for figure in ("ranking-metrics.png", "cmc.png", "confusion.png", "score-distribution.png"):
        display(Image(filename=str(folder / figure)))

display(Markdown("### Auditoría visual del preprocesamiento público"))
for figure in sorted((REPORT / "public_test").glob("preprocessing-*.png")):
    display(Image(filename=str(figure)))
display(Markdown(
    f"MLflow: [run PetTrace]({publication['url']}) · "
    f"run público `{publication['public_run_id']}`"
))

### Discusión y limitaciones

E2 aísla una intervención de preprocesamiento, no aprendizaje de representaciones. Un mejor resultado agregado no implica que todo recorte sea correcto; por eso el reporte conserva detecciones ausentes, cajas inválidas, desacuerdos de especie, área retenida y cambios de ranking por consulta.

El detector fue preentrenado en COCO y no optimizado con estos tests. La EER sigue siendo descriptiva y las fotografías de un mismo animal no son observaciones independientes. Las diferencias entre el test público y PetTrace no son causales porque cambian identidades, galerías y condiciones de captura.

Las salidas ejecutadas del E2 histórico sobre el benchmark piloto público v1 y PetTrace v4 se preservan en `notebooks/archive/e2_yolo_crop_zero_shot_v1_v4.ipynb`; no constituyen resultados de esta reproducción.